# Feature Engineering vs. Raw Features with XGBoost

Concrete example of how feature engineering can improve an XGBoost model's test accuracy.

The setup here is artifical: we simulate a small "loan default" dataset where we intentionaly craft the target to depend on:
1. The debt-to-income ratio (`debt / income`) — this is a ratio, which axis-aligned tree splits struggle to learn from the two raw columns.
2. Whether the customer signed up on a weekend — this is hidden inside a raw `day` feature. A tree can't discover `day % 7` on its own.

We train the same XGBoost model twice: once on raw columns, once with two engineered features added.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

RANDOM_STATE = 42

## Create the synthetic dataset

The label is driven by `debt/income` and a weekend effect, plus 10 pure-noise columns to make things realistic.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 2000

income     = rng.uniform(20_000, 150_000, n)   # annual income ($)
debt       = rng.uniform(1_000, 100_000, n)    # total debt ($)
signup_day = rng.integers(0, 365 * 6, n)       # days since product launch

# hidden variables that we'll say our target actually depends on
dti        = debt / income                     # debt-to-income ratio
is_weekend = np.isin(signup_day % 7, [5, 6])   # signed up on a weekend?

# generating our target
logit = 4 * (dti - 0.55) + 4 * is_weekend - 4 * 0.28 + rng.normal(0, 0.5, n)
y = (logit > 0).astype(int)                    # 1 = default

# generating our full set of features
X_raw = pd.DataFrame({'income': income, 'debt': debt, 'signup_day': signup_day})
# and add in a variety of irrelevant features (noise) 
for i in range(10):                            
    X_raw[f'noise_{i}'] = rng.normal(size=n)

X_raw.head()

In [ ]:
X_raw.describe()

In [ ]:
np.unique(y), np.mean(y)

## Baseline: XGBoost on raw features

In [ ]:
def evaluate(X, y, label):
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                              test_size=0.3, 
                                              random_state=RANDOM_STATE, 
                                              stratify=y)
    model = XGBClassifier(n_estimators=50, 
                          max_depth=3, 
                          learning_rate=0.1,
                          random_state=RANDOM_STATE, 
                          eval_metric='logloss')
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{label:<28} test accuracy: {acc:.4f}")
    return acc, model

acc_raw, model_raw = evaluate(X_raw, y, "Raw features")

## Feature engineering

Two lines of domain knowledge:
- `debt_to_income` — expose the ratio directly.
- `is_weekend` — decode the calendar structure hidden in `signup_day`.

In [ ]:
X_engineered = X_raw.copy()
X_engineered['debt_to_income'] = X_engineered['debt'] / X_engineered['income']
X_engineered['is_weekend']     = np.isin(X_engineered['signup_day'] % 7, [5, 6]).astype(int)

acc_engineered, model_engineered = evaluate(X_engineered, y, "Raw + engineered features")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Accuracy comparison
bars = ax1.bar(['Raw', 'Engineered'], 
               [acc_raw, acc_engineered])
ax1.set_ylim(0.5, 1.0)
ax1.set_ylabel('Test accuracy')
ax1.set_title('Accuracy')
for b, v in zip(bars, [acc_raw, acc_engineered]):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}', ha='center')

# Feature importance of the engineered model
imp = pd.Series(model_engineered.feature_importances_, index=X_engineered.columns).sort_values()
imp.tail(6).plot.barh(ax=ax2)
ax2.set_title('Top feature importances (engineered model)')

plt.tight_layout()
plt.show()

## Results

- Ratios: a tree splits one feature at a time (`debt < c` or `income < c`), so approximating a diagonal boundary like `debt/income > 0.55` needs many staircase-like splits. Handing it the ratio turns that into a single clean split.
- Cyclical/date structure: nothing in `signup_day`'s raw values reveals `day % 7`. The model literally cannot recover the weekend effect without help.
- The engineered model concentrates on the two new features (right chart of feature importances) — this signal was missing for the raw model.

Even powerful models like XGBoost only search over the representation you give them. A bit of domain-informed feature engineering can be worth more than any amount of hyperparameter tuning.